# 10 量化、显存和速度

目标：理解 dtype 降精度和 int8/int4 量化的区别，并用同一段 prompt 对比显存、dtype 和生成速度。


## 1. 安装依赖


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

base = Path.cwd()
requirements_path = base / "requirements.txt"
advanced_requirements_path = base / "advanced" / "requirements-advanced.txt"

if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

if not advanced_requirements_path.exists():
    if Path("requirements-advanced.txt").exists():
        advanced_requirements_path = Path("requirements-advanced.txt")
    else:
        advanced_requirements_path = Path("../advanced/requirements-advanced.txt")


def read_requirements(path):
    if not path.exists():
        return []
    rows = []
    for line in path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#"):
            rows.append(line)
    return rows


requirements = read_requirements(requirements_path) + read_requirements(advanced_requirements_path)

# ROCm/PyTorch builds are usually installed from a ROCm-specific wheel index.
# Avoid replacing a working ROCm torch with a generic pip build from this notebook.
if importlib.util.find_spec("torch") is not None:
    import torch

    if getattr(torch.version, "hip", None):
        requirements = [req for req in requirements if req.split("==")[0].split(">=")[0] != "torch"]
        print("Detected ROCm PyTorch:", torch.__version__, "HIP:", torch.version.hip)
        print("Skipping torch from requirements to keep the ROCm build intact.")

print("requirements:", requirements_path)
print("advanced requirements:", advanced_requirements_path)
print("installing:", requirements)
if requirements:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *requirements])


## 2. 模型与工具函数


In [ ]:
import gc
import os
from pathlib import Path
from time import perf_counter

import torch
from modelscope import snapshot_download
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

MODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
MODEL_SOURCE = os.getenv("MODEL_SOURCE", "modelscope").lower()
GPU_AVAILABLE = torch.cuda.is_available()
BACKEND = "rocm" if getattr(torch.version, "hip", None) else ("cuda" if GPU_AVAILABLE else "cpu")


def resolve_model_path(model_id):
    if Path(model_id).exists():
        return model_id
    if MODEL_SOURCE == "modelscope":
        return snapshot_download(model_id)
    return model_id


def cleanup(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def gpu_memory(label=""):
    if not torch.cuda.is_available():
        print(label, "gpu unavailable")
        return
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    print(f"{label} allocated={allocated:.2f}GB reserved={reserved:.2f}GB peak={peak:.2f}GB")


# In PyTorch ROCm builds, AMD GPUs are still exposed through torch.cuda APIs.
cuda_memory = gpu_memory

MODEL_PATH = resolve_model_path(MODEL_ID)
print("MODEL_ID =", MODEL_ID)
print("MODEL_PATH =", MODEL_PATH)
print("torch =", torch.__version__)
print("backend =", BACKEND)
print("hip =", getattr(torch.version, "hip", None))
print("gpu available =", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu =", torch.cuda.get_device_name(0))
    print("bf16 supported =", torch.cuda.is_bf16_supported())


## 3. 先看模型结构和理论权重显存

这里只估算权重本身，不包含 KV cache、activation、GPU/ROCm workspace 和框架开销。


In [ ]:
config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("model_type:", config.model_type)
print("hidden_size:", getattr(config, "hidden_size", "unknown"))
print("num_hidden_layers:", getattr(config, "num_hidden_layers", "unknown"))
print("num_attention_heads:", getattr(config, "num_attention_heads", "unknown"))
print("num_key_value_heads:", getattr(config, "num_key_value_heads", "unknown"))


def estimate_weight_memory(num_params):
    rows = []
    for name, bytes_per_param in [
        ("float32", 4),
        ("float16/bfloat16", 2),
        ("int8", 1),
        ("int4", 0.5),
    ]:
        rows.append((name, num_params * bytes_per_param / 1024**3))
    return rows


## 4. dtype 对比：FP32、FP16、BF16

`torch_dtype` 仍然是浮点表示。它简单稳定，通常是部署首选。


In [ ]:
def load_and_time_dtype(torch_dtype):
    cleanup()
    start = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch_dtype,
        device_map="auto",
        trust_remote_code=True,
    )
    load_seconds = perf_counter() - start
    actual_dtype = next(model.parameters()).dtype
    num_params = sum(p.numel() for p in model.parameters())

    print("requested dtype:", torch_dtype)
    print("actual dtype:", actual_dtype)
    print("parameters:", f"{num_params/1e6:.1f}M")
    print("load seconds:", round(load_seconds, 3))
    for name, gb in estimate_weight_memory(num_params):
        print(f"estimated {name}: {gb:.2f}GB")
    gpu_memory("after load")
    return tokenizer, model


def benchmark_generation(tokenizer, model, prompt_text="用三句话解释量化对大模型部署的好处。", max_new_tokens=120):
    messages = [{"role": "user", "content": prompt_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = perf_counter()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = perf_counter() - start

    new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    print("new tokens:", len(new_tokens))
    print("seconds:", round(elapsed, 3))
    print("tokens/s:", round(len(new_tokens) / elapsed, 2))
    print(tokenizer.decode(new_tokens, skip_special_tokens=True))
    gpu_memory("after generate")


In [ ]:
dtype_tests = ["auto"]
if GPU_AVAILABLE:
    dtype_tests.extend([torch.float16, torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16])
dtype_tests.append(torch.float32)

for dtype in dtype_tests:
    print("=" * 80)
    tokenizer, model = load_and_time_dtype(dtype)
    benchmark_generation(tokenizer, model, max_new_tokens=80)
    cleanup(tokenizer, model)


## 5. 8bit / 4bit 量化

量化会让权重以 int8/int4 常驻显存。计算时通常由 kernel 分块反量化或边反量化边计算，不会把整个模型完整还原成 FP16 常驻显存。


In [ ]:
from transformers import BitsAndBytesConfig


def load_quantized(load_in_8bit=False, load_in_4bit=False):
    if BACKEND == "rocm":
        print("ROCm backend detected. bitsandbytes 8bit/4bit requires a ROCm-compatible bitsandbytes build; failures here usually mean the quantization kernel is unsupported, not that the GPU is unavailable.")
    cleanup()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quant_config = BitsAndBytesConfig(
        load_in_8bit=load_in_8bit,
        load_in_4bit=load_in_4bit,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print("8bit:", load_in_8bit, "4bit:", load_in_4bit)
    print("device map:", getattr(model, "hf_device_map", None))
    gpu_memory("after quantized load")
    return tokenizer, model


In [ ]:
# 如果 bitsandbytes 或 GPU/ROCm 环境不支持，这个单元可能报错；这本身也是部署兼容性测试的一部分。
for kwargs in [{"load_in_8bit": True}, {"load_in_4bit": True}]:
    print("=" * 80)
    try:
        tokenizer, model = load_quantized(**kwargs)
        benchmark_generation(tokenizer, model, max_new_tokens=80)
        cleanup(tokenizer, model)
    except Exception as exc:
        print("quantization failed:", type(exc).__name__, exc)


## 面试总结

- dtype 降精度：FP32 -> FP16/BF16，简单稳定，显存约减半。
- 量化：FP16/FP32 -> INT8/INT4，显存更低，但依赖量化格式、scale、kernel 和硬件支持。
- 速度不一定更快：小模型可能因为反量化开销变慢，大模型可能因为显存带宽压力降低而变快。
